# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vikasbit/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

I will prioritize pages that have meaningful search demand but show weaker click performance relative to their search position.

The score will combine:
- GSC impressions: higher impressions increase priority because more search demand is available.
- GSC average position: pages already ranking relatively well but receiving fewer clicks are stronger refresh candidates.
- GSC CTR: lower CTR increases priority when the page has enough impressions to make the signal meaningful.

The rule uses only information available at the decision point. It does not use future outcomes or label-derived fields.

### Reason codes

- CTR_OPPORTUNITY — the page has meaningful impressions and relatively weak CTR.
- HIGH_DEMAND — the page has high impressions and therefore has larger potential search exposure.
- RANKING_OPPORTUNITY — the page has a useful search position but weak click performance.
- LOWER_PRIORITY — the available signals do not indicate a strong refresh opportunity.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [19]:
!pip -q install huggingface_hub

In [22]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN").strip()

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Hugging Face connection refreshed.")

Hugging Face connection refreshed.


In [24]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Length:", len(HF_TOKEN))
print("Number of lines:", len(HF_TOKEN.splitlines()))
print("Starts correctly:", HF_TOKEN.startswith("hf_"))
print("Contains newline:", "\n" in HF_TOKEN)

Length: 76
Number of lines: 2
Starts correctly: True
Contains newline: True


In [36]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN").splitlines()[0].strip()

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Hugging Face connection refreshed.")

Hugging Face connection refreshed.


In [26]:
MARCH_FACT = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
"""

print(con.sql(f"""
SELECT COUNT(*) AS march_rows
FROM {MARCH_FACT}
""").df())

   march_rows
0     9841378


In [37]:
q2 = con.sql(f"""
WITH page_level AS (
    SELECT
        MAX(report_date) AS report_date,
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN CAST(SUM(gsc_clicks) AS DOUBLE)
                 / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr,

        AVG(gsc_avg_position) AS gsc_avg_position

    FROM {MARCH_FACT}

    WHERE gsc_data_available IS TRUE
      AND gsc_impressions IS NOT NULL
      AND gsc_impressions > 0

    GROUP BY
        client_hash_id,
        content_hash_id
),

scored AS (
    SELECT
        *,
        PERCENT_RANK() OVER (
            ORDER BY LN(1 + gsc_impressions)
        ) AS demand_score,

        1 - PERCENT_RANK() OVER (
            ORDER BY ctr
        ) AS ctr_opportunity,

        1 - PERCENT_RANK() OVER (
            ORDER BY gsc_avg_position NULLS LAST
        ) AS position_score

    FROM page_level
),

final_scores AS (
    SELECT
        *,
        ROUND(
            100 * (
                0.45 * demand_score
                + 0.35 * ctr_opportunity
                + 0.20 * position_score
            ),
            2
        ) AS action_score

    FROM scored
),

labelled AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        ROUND(ctr, 4) AS ctr,
        ROUND(gsc_avg_position, 2) AS gsc_avg_position,
        action_score,

        CASE
            WHEN ctr_opportunity >= 0.70
             AND demand_score >= 0.50
                THEN 'CTR_OPPORTUNITY'
            WHEN demand_score >= 0.80
                THEN 'HIGH_DEMAND'
            WHEN position_score >= 0.70
                THEN 'RANKING_OPPORTUNITY'
            ELSE 'LOWER_PRIORITY'
        END AS reason_code,

        CASE
            WHEN ctr_opportunity >= 0.70
             AND demand_score >= 0.50
                THEN 'REFRESH_CONTENT'
            WHEN demand_score >= 0.80
                THEN 'REVIEW_HIGH_DEMAND'
            WHEN position_score >= 0.70
                THEN 'REVIEW_RANKING'
            ELSE 'MONITOR'
        END AS action

    FROM final_scores
)

SELECT
    *,
    ROW_NUMBER() OVER (
        ORDER BY action_score DESC
    ) AS rank
FROM labelled
ORDER BY rank
""").df()

print("Rows in ranked queue:", len(q2))
display(q2.head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in ranked queue: 176738


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,action_score,reason_code,action,rank
0,2026-03-31,client_a80fca3f171ed1de,content_fa17add7836d36c3,12588.0,0.0,0.0,1.90,98.15,CTR_OPPORTUNITY,REFRESH_CONTENT,1
1,2026-03-31,client_73cda7b4e4f265ea,content_d397987113cb84a0,9887.0,0.0,0.0,1.95,97.69,CTR_OPPORTUNITY,REFRESH_CONTENT,2
2,2026-03-31,client_e547b89c05043229,content_83167156f76e33e5,6827.0,0.0,0.0,1.14,97.18,CTR_OPPORTUNITY,REFRESH_CONTENT,3
3,2026-03-31,client_23a62021009f63c4,content_a27b382f00aa75c6,7736.0,0.0,0.0,2.26,96.80,CTR_OPPORTUNITY,REFRESH_CONTENT,4
4,2026-03-31,client_23a62021009f63c4,content_fe8baba849843607,14813.0,0.0,0.0,3.43,96.67,CTR_OPPORTUNITY,REFRESH_CONTENT,5
5,2026-03-31,client_e547b89c05043229,content_713b157e9c77690a,24908.0,0.0,0.0,4.01,96.24,CTR_OPPORTUNITY,REFRESH_CONTENT,6
6,2026-03-31,client_fef1a8f436438636,content_1bc8782404e3b132,5792.0,0.0,0.0,2.26,95.99,CTR_OPPORTUNITY,REFRESH_CONTENT,7
7,2026-03-31,client_62f4a7e64f5e0096,content_db01d94616cdb80d,4202.0,0.0,0.0,0.91,95.49,CTR_OPPORTUNITY,REFRESH_CONTENT,8
8,2026-03-31,client_20259bd6705d81d4,content_9cec93fc44a7ab41,4742.0,0.0,0.0,1.85,95.47,CTR_OPPORTUNITY,REFRESH_CONTENT,9
9,2026-03-31,client_e547b89c05043229,content_ecc27d32010b18e0,3900.0,0.0,0.0,0.55,95.26,CTR_OPPORTUNITY,REFRESH_CONTENT,10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [38]:
# 3. Top-20 review

top20 = q2.head(20).copy()

top20["why_selected"] = top20.apply(
    lambda r: (
        f"Score {r['action_score']}/100; "
        f"impressions={r['gsc_impressions']:,}, "
        f"CTR={r['ctr']:.4f}, "
        f"average position={r['gsc_avg_position']:.2f}, "
        f"reason={r['reason_code']}."
    ),
    axis=1
)

top20["what_could_make_it_wrong"] = top20.apply(
    lambda r: (
        "The low CTR may be expected for the query intent, "
        "the page may already be intentionally optimized, "
        "or the aggregated monthly signals may hide important page/query differences."
    ),
    axis=1
)

review = top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "action_score",
        "why_selected",
        "what_could_make_it_wrong"
    ]
]

display(review)


,rank,client_hash_id,content_hash_id,action,reason_code,action_score,why_selected,what_could_make_it_wrong
0,1,client_a80fca3f171ed1de,content_fa17add7836d36c3,REFRESH_CONTENT,CTR_OPPORTUNITY,98.15,"Score 98.15/100; impressions=12,588.0, CTR=0.0...",The low CTR may be expected for the query inte...
1,2,client_73cda7b4e4f265ea,content_d397987113cb84a0,REFRESH_CONTENT,CTR_OPPORTUNITY,97.69,"Score 97.69/100; impressions=9,887.0, CTR=0.00...",The low CTR may be expected for the query inte...
2,3,client_e547b89c05043229,content_83167156f76e33e5,REFRESH_CONTENT,CTR_OPPORTUNITY,97.18,"Score 97.18/100; impressions=6,827.0, CTR=0.00...",The low CTR may be expected for the query inte...
3,4,client_23a62021009f63c4,content_a27b382f00aa75c6,REFRESH_CONTENT,CTR_OPPORTUNITY,96.80,"Score 96.8/100; impressions=7,736.0, CTR=0.000...",The low CTR may be expected for the query inte...
4,5,client_23a62021009f63c4,content_fe8baba849843607,REFRESH_CONTENT,CTR_OPPORTUNITY,96.67,"Score 96.67/100; impressions=14,813.0, CTR=0.0...",The low CTR may be expected for the query inte...
5,6,client_e547b89c05043229,content_713b157e9c77690a,REFRESH_CONTENT,CTR_OPPORTUNITY,96.24,"Score 96.24/100; impressions=24,908.0, CTR=0.0...",The low CTR may be expected for the query inte...
6,7,client_fef1a8f436438636,content_1bc8782404e3b132,REFRESH_CONTENT,CTR_OPPORTUNITY,95.99,"Score 95.99/100; impressions=5,792.0, CTR=0.00...",The low CTR may be expected for the query inte...
7,8,client_62f4a7e64f5e0096,content_db01d94616cdb80d,REFRESH_CONTENT,CTR_OPPORTUNITY,95.49,"Score 95.49/100; impressions=4,202.0, CTR=0.00...",The low CTR may be expected for the query inte...
8,9,client_20259bd6705d81d4,content_9cec93fc44a7ab41,REFRESH_CONTENT,CTR_OPPORTUNITY,95.47,"Score 95.47/100; impressions=4,742.0, CTR=0.00...",The low CTR may be expected for the query inte...
9,10,client_e547b89c05043229,content_ecc27d32010b18e0,REFRESH_CONTENT,CTR_OPPORTUNITY,95.26,"Score 95.26/100; impressions=3,900.0, CTR=0.00...",The low CTR may be expected for the query inte...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [39]:
# 4. Inspect a few weak picks

weak_picks = q2.tail(10).copy()

weak_picks["why_not_prioritized"] = weak_picks.apply(
    lambda r: (
        f"Score {r['action_score']}/100 with action "
        f"'{r['action']}'. The available signals did not "
        f"produce a strong baseline opportunity."
    ),
    axis=1
)

display(
    weak_picks[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "action_score",
            "reason_code",
            "action",
            "why_not_prioritized"
        ]
    ]
)


,rank,client_hash_id,content_hash_id,action_score,reason_code,action,why_not_prioritized
176728,176729,client_3ffa76342f366962,content_48ca00f9305fa67e,2.22,LOWER_PRIORITY,MONITOR,Score 2.22/100 with action 'MONITOR'. The avai...
176729,176730,client_3ffa76342f366962,content_c7625dad415177be,2.13,LOWER_PRIORITY,MONITOR,Score 2.13/100 with action 'MONITOR'. The avai...
176730,176731,client_3ffa76342f366962,content_1bd7fde32cf86a20,1.96,LOWER_PRIORITY,MONITOR,Score 1.96/100 with action 'MONITOR'. The avai...
176731,176732,client_3ffa76342f366962,content_06ea8e909635b4b2,1.96,LOWER_PRIORITY,MONITOR,Score 1.96/100 with action 'MONITOR'. The avai...
176732,176733,client_f623b01661d4bfe4,content_507653893440072c,1.66,LOWER_PRIORITY,MONITOR,Score 1.66/100 with action 'MONITOR'. The avai...
176733,176734,client_f623b01661d4bfe4,content_b11dddc3f12c8b25,1.53,LOWER_PRIORITY,MONITOR,Score 1.53/100 with action 'MONITOR'. The avai...
176734,176735,client_3197e6291363b4db,content_936a12b761e9bed7,1.41,LOWER_PRIORITY,MONITOR,Score 1.41/100 with action 'MONITOR'. The avai...
176735,176736,client_3ffa76342f366962,content_5b12e81bfdba5de5,1.25,LOWER_PRIORITY,MONITOR,Score 1.25/100 with action 'MONITOR'. The avai...
176736,176737,client_f623b01661d4bfe4,content_b589c670c71a5db2,0.97,LOWER_PRIORITY,MONITOR,Score 0.97/100 with action 'MONITOR'. The avai...
176737,176738,client_cd12bcfd98942aa1,content_9a5496694a583150,0.97,LOWER_PRIORITY,MONITOR,Score 0.97/100 with action 'MONITOR'. The avai...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.